# TNG300-1 environment robustness: thresholds, projection, and Lagrangian scale

This notebook tests whether the strong connection between the $z\simeq8$ large-scale environment and the snapshot-99 descendant FoF-host mass is robust to tracer thresholds and observational projection. It also compares the preferred environment scale with the descendant's Lagrangian radius.

The existing `tng300_z8_M11_descendants.csv` is reused, so SubLink tracking is not repeated. Every $M_0$ value remains the `Group_M_Crit200` of the descendant subhalo's snapshot-99 FoF host.

## 1. Imports and constants

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from scipy.spatial import cKDTree
from tqdm.auto import tqdm
from IPython.display import display, Markdown

import illustris_python as il

plt.rcParams.update({
    "figure.figsize": (7.4, 4.9),
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

### Paths, cosmology, and analysis grid

Positions are converted from comoving kpc/$h$ to cMpc; masses are converted from $10^{10}M_\odot/h$ to physical $M_\odot$. The TNG cosmology is read from the group-catalog header. The $10^{7.5}M_\odot$ galaxy threshold is intentionally included as a stress test but is not expected to be complete in TNG300-1: it corresponds to only a few initial baryonic resolution elements and must not be interpreted on equal footing with higher thresholds.

In [ ]:
base_candidates = [
    Path("/home/tnguser/sims.TNG/L205n2500TNG/output"),
    Path("../sims.TNG/TNG300-1/output").resolve(),
    Path("../sims.TNG/L205n2500TNG/output").resolve(),
]
basePath = next((p for p in base_candidates if p.is_dir()), None)
if basePath is None:
    raise FileNotFoundError("Set basePath to the TNG300-1 output directory.")

descendant_csv = Path.cwd() / "tng300_z8_M11_descendants.csv"
summary_csv = Path.cwd() / "tng300_environment_robustness_summary.csv"

snap_z8 = 8
snap_z0 = 99
logM_min, logM_max = 10.8, 11.2
R_sphere_cMpc = np.array([1, 2, 3, 5, 8, 10, 15, 20], dtype=float)
logMhalo_cuts = np.array([9.5, 10.0, 10.5, 11.0])
logMstar_cuts = np.array([7.5, 8.0, 8.5, 9.0])
R_proj_cMpc = np.array([2, 3, 5, 8, 10, 15], dtype=float)
L_los_cMpc = np.array([10, 20, 40, 80], dtype=float)
cylinder_logMstar_cut = 8.0
require_subhalo_flag = True
exclude_target = True
query_chunk_size = 256
sfr_floor = 1.0e-3
random_state = 42

header_z8 = il.groupcat.loadHeader(str(basePath), snap_z8)
z_z8 = float(header_z8["Redshift"])
h = float(header_z8["HubbleParam"])
Omega_m = float(header_z8["Omega0"])
box_size_cMpc = float(header_z8["BoxSize"]) / (1000.0 * h)

assert np.isclose(h, 0.6774, atol=1e-4, rtol=0)
assert np.all(np.diff(R_sphere_cMpc) > 0)
assert max(R_sphere_cMpc.max(), R_proj_cMpc.max(), L_los_cMpc.max()/2) < box_size_cMpc/2

print(f"basePath={basePath}")
print(f"snapshot={snap_z8}, z={z_z8:.6f}, h={h:.4f}, Omega_m={Omega_m:.4f}")
print(f"box={box_size_cMpc:.3f} cMpc")
print(f"sphere radii={R_sphere_cMpc.tolist()} cMpc")
print(f"halo cuts={logMhalo_cuts.tolist()}, stellar cuts={logMstar_cuts.tolist()}")

## 2. Load the existing target and descendant catalog

In [ ]:
required_columns = [
    "GroupID_z8", "SubhaloID_z8", "M200c_z8", "logM200c_z8",
    "GroupID_z0", "SubhaloID_z0", "M200c_z0", "logM200c_z0",
]
if not descendant_csv.is_file():
    raise FileNotFoundError(f"Missing {descendant_csv}")
raw = pd.read_csv(descendant_csv)
missing = sorted(set(required_columns) - set(raw.columns))
if missing:
    raise ValueError(f"Missing descendant columns: {missing}")

finite = raw[required_columns].notna().all(axis=1)
selected = (raw["logM200c_z8"] > logM_min) & (raw["logM200c_z8"] < logM_max)
targets = raw.loc[finite & selected, required_columns].copy().reset_index(drop=True)
for column in ["GroupID_z8", "SubhaloID_z8", "GroupID_z0", "SubhaloID_z0"]:
    targets[column] = targets[column].astype(np.int64)

print(f"Catalog rows={len(raw):,}; finite tracked={finite.sum():,}; targets={len(targets):,}")
display(targets.head())

### Load snapshot-8 halo and galaxy tracer catalogs once

PartType4 is the stellar/wind component. `SubhaloFlag == 1` is used by default to retain cosmological subhalos. Target-to-catalog IDs and masses are explicitly checked.

In [ ]:
halos = il.groupcat.loadHalos(str(basePath), snap_z8, fields=["GroupPos", "Group_M_Crit200"])
group_pos = np.mod(np.asarray(halos["GroupPos"], dtype=float) / (1000.0*h), box_size_cMpc)
group_mass = np.asarray(halos["Group_M_Crit200"], dtype=float) * 1.0e10 / h

subs = il.groupcat.loadSubhalos(
    str(basePath), snap_z8,
    fields=["SubhaloPos", "SubhaloMassType", "SubhaloGrNr", "SubhaloFlag", "SubhaloSFR"],
)
sub_pos = np.mod(np.asarray(subs["SubhaloPos"], dtype=float) / (1000.0*h), box_size_cMpc)
sub_mass_type = np.asarray(subs["SubhaloMassType"], dtype=float)
sub_mstar = sub_mass_type[:, 4] * 1.0e10 / h
sub_grnr = np.asarray(subs["SubhaloGrNr"], dtype=np.int64)
sub_flag = np.asarray(subs["SubhaloFlag"], dtype=bool)
sub_sfr = np.asarray(subs["SubhaloSFR"], dtype=float)

target_group_ids = targets["GroupID_z8"].to_numpy(dtype=np.int64)
target_sub_ids = targets["SubhaloID_z8"].to_numpy(dtype=np.int64)
target_pos = group_pos[target_group_ids]

assert np.all(sub_grnr[target_sub_ids] == target_group_ids)
mass_error = np.abs(np.log10(group_mass[target_group_ids]) - targets["logM200c_z8"].to_numpy())
assert np.nanmax(mass_error) < 1e-4

targets["MstarCentral_z8"] = sub_mstar[target_sub_ids]
targets["logMstarCentral_z8"] = np.where(
    targets["MstarCentral_z8"] > 0, np.log10(targets["MstarCentral_z8"]), np.nan
)
targets["SFRCentral_z8"] = sub_sfr[target_sub_ids]
targets["logSFRCentral_z8"] = np.log10(targets["SFRCentral_z8"] + sfr_floor)

halo_min_mask = np.isfinite(group_mass) & (group_mass > 10**logMhalo_cuts.min())
halo_ids = np.flatnonzero(halo_min_mask)
galaxy_min_mask = np.isfinite(sub_mstar) & (sub_mstar > 10**logMstar_cuts.min())
if require_subhalo_flag:
    galaxy_min_mask &= sub_flag
galaxy_ids = np.flatnonzero(galaxy_min_mask)

print(f"FoF groups={len(group_mass):,}; halo candidates above min cut={len(halo_ids):,}")
print(f"Subhalos={len(sub_mstar):,}; galaxy candidates above min cut={len(galaxy_ids):,}")
print(f"Maximum target mass mismatch={np.nanmax(mass_error):.2e} dex")

## 3. Reusable utility functions

In [ ]:
def periodic_delta(pos1, pos2, boxsize):
    delta = np.asarray(pos1, dtype=float) - np.asarray(pos2, dtype=float)
    return (delta + boxsize/2.0) % boxsize - boxsize/2.0


def periodic_distance(pos1, pos2, boxsize):
    return np.sqrt(np.sum(periodic_delta(pos1, pos2, boxsize)**2, axis=-1))


edge_a = np.array([0.2, 10.0, 20.0])
edge_b = np.array([box_size_cMpc-0.2, 10.0, 20.0])
assert np.isclose(periodic_distance(edge_a, edge_b, box_size_cMpc), 0.4)
assert set(cKDTree(np.vstack([edge_a, edge_b]), boxsize=box_size_cMpc).query_ball_point(edge_a, 0.5)) == {0, 1}
print("Periodic minimum-image boundary test passed.")


def safe_spearman(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 3 or np.unique(x[valid]).size < 2 or np.unique(y[valid]).size < 2:
        return np.nan, np.nan, int(valid.sum())
    rho, p = spearmanr(x[valid], y[valid])
    return float(rho), float(p), int(valid.sum())


def percentile_summary(values, label="sample"):
    values = np.asarray(pd.Series(values).dropna(), float)
    if values.size == 0:
        return pd.Series({"Label": label, "N": 0, "Median": np.nan, "P16": np.nan,
                          "P84": np.nan, "W68": np.nan})
    p16, med, p84 = np.percentile(values, [16, 50, 84])
    return pd.Series({"Label": label, "N": values.size, "Median": med,
                      "P16": p16, "P84": p84, "W68": p84-p16})


def rank_tertiles(series):
    return pd.qcut(series.rank(method="first"), 3, labels=["low", "middle", "high"])

### Multi-threshold periodic sphere search

For each tracer type, a single periodic KD-tree is built from the lowest-threshold catalog. Each target is queried only to the maximum radius; distances are calculated once, then all four thresholds and eight cumulative radii are evaluated from that same neighbor list.

In [ ]:
def multi_threshold_sphere(
    target_positions, catalog_positions, catalog_masses,
    catalog_ids, target_ids, log_thresholds, radii, boxsize,
    chunk_size=256, description="sphere search",
):
    target_positions = np.asarray(target_positions, float)
    catalog_positions = np.asarray(catalog_positions, float)
    catalog_masses = np.asarray(catalog_masses, float)
    catalog_ids = np.asarray(catalog_ids, np.int64)
    target_ids = np.asarray(target_ids, np.int64)
    log_thresholds = np.asarray(log_thresholds, float)
    radii = np.asarray(radii, float)

    counts = np.zeros((len(target_positions), len(log_thresholds), len(radii)), dtype=np.int32)
    mass_sums = np.zeros_like(counts, dtype=float)
    tree = cKDTree(catalog_positions, boxsize=boxsize)

    starts = range(0, len(target_positions), chunk_size)
    for start in tqdm(starts, total=(len(target_positions)+chunk_size-1)//chunk_size, desc=description):
        stop = min(start+chunk_size, len(target_positions))
        try:
            lists = tree.query_ball_point(target_positions[start:stop], radii[-1], workers=-1)
        except TypeError:
            lists = tree.query_ball_point(target_positions[start:stop], radii[-1])

        for local_i, neighbors in enumerate(lists):
            i = start + local_i
            idx = np.asarray(neighbors, dtype=np.int64)
            if exclude_target and idx.size:
                idx = idx[catalog_ids[idx] != target_ids[i]]
            if idx.size == 0:
                continue
            distances = periodic_distance(catalog_positions[idx], target_positions[i], boxsize)
            masses = catalog_masses[idx]
            for t, threshold in enumerate(log_thresholds):
                use = masses > 10**threshold
                if not np.any(use):
                    continue
                d = distances[use]
                w = masses[use]
                order = np.argsort(d)
                d, w = d[order], w[order]
                cumulative_mass = np.cumsum(w)
                k = np.searchsorted(d, radii, side="left")
                counts[i, t] = k
                positive = k > 0
                mass_sums[i, t, positive] = cumulative_mass[k[positive]-1]
    return counts, mass_sums

### Periodic observational-cylinder search

The line of sight is the $z$ axis. A periodic 2D KD-tree finds all candidates within the largest projected radius in $(x,y)$; a minimum-image $|\Delta z|<L_{\rm los}/2$ cut then supplies each cylinder depth.

In [ ]:
def periodic_cylinders(
    target_positions, catalog_positions, catalog_weights,
    catalog_ids, target_ids, projected_radii, los_depths, boxsize,
    chunk_size=256, description="cylinder search",
):
    projected_radii = np.asarray(projected_radii, float)
    los_depths = np.asarray(los_depths, float)
    tree_xy = cKDTree(catalog_positions[:, :2], boxsize=[boxsize, boxsize])
    counts = np.zeros((len(target_positions), len(los_depths), len(projected_radii)), dtype=np.int32)
    weighted_sums = np.zeros_like(counts, dtype=float)

    starts = range(0, len(target_positions), chunk_size)
    for start in tqdm(starts, total=(len(target_positions)+chunk_size-1)//chunk_size, desc=description):
        stop = min(start+chunk_size, len(target_positions))
        try:
            lists = tree_xy.query_ball_point(target_positions[start:stop, :2], projected_radii[-1], workers=-1)
        except TypeError:
            lists = tree_xy.query_ball_point(target_positions[start:stop, :2], projected_radii[-1])

        for local_i, neighbors in enumerate(lists):
            i = start + local_i
            idx = np.asarray(neighbors, dtype=np.int64)
            if exclude_target and idx.size:
                idx = idx[catalog_ids[idx] != target_ids[i]]
            if idx.size == 0:
                continue
            delta = periodic_delta(catalog_positions[idx], target_positions[i], boxsize)
            r_perp = np.sqrt(delta[:, 0]**2 + delta[:, 1]**2)
            abs_los = np.abs(delta[:, 2])
            weights = catalog_weights[idx]
            for l, depth in enumerate(los_depths):
                use = abs_los < depth/2.0
                if not np.any(use):
                    continue
                r, w = r_perp[use], weights[use]
                order = np.argsort(r)
                r, w = r[order], w[order]
                cumulative_weight = np.cumsum(w)
                k = np.searchsorted(r, projected_radii, side="left")
                counts[i, l] = k
                positive = k > 0
                weighted_sums[i, l, positive] = cumulative_weight[k[positive]-1]
    return counts, weighted_sums

## 4. Halo-threshold dependence

In [ ]:
halo_counts, halo_mass_sums = multi_threshold_sphere(
    target_pos, group_pos[halo_ids], group_mass[halo_ids],
    halo_ids, target_group_ids, logMhalo_cuts, R_sphere_cMpc, box_size_cMpc,
    query_chunk_size, "Halo threshold grid",
)

halo_rows = []
for t, threshold in enumerate(logMhalo_cuts):
    for r, radius in enumerate(R_sphere_cMpc):
        rho_n, p_n, n_n = safe_spearman(halo_counts[:, t, r], targets["logM200c_z0"])
        rho_m, p_m, n_m = safe_spearman(halo_mass_sums[:, t, r], targets["logM200c_z0"])
        halo_rows.append({
            "log_threshold": threshold, "R_cMpc": radius,
            "rho_count": rho_n, "rho_mass": rho_m,
            "p_count": p_n, "p_mass": p_m, "N": n_n,
            "zero_fraction": np.mean(halo_counts[:, t, r] == 0),
        })
halo_correlations = pd.DataFrame(halo_rows)
display(halo_correlations.round(4))

### Halo-threshold scale dependence

Both neighbor count and total neighbor $M_{200c}$ are shown. Zero-neighbor fractions diagnose loss of dynamic range at high thresholds and small apertures.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.7), sharey=True)
for threshold in logMhalo_cuts:
    part = halo_correlations[halo_correlations["log_threshold"] == threshold]
    axes[0].plot(part["R_cMpc"], part["rho_count"], "o-", label=rf"$10^{{{threshold:g}}}M_\odot$")
    axes[1].plot(part["R_cMpc"], part["rho_mass"], "o-", label=rf"$10^{{{threshold:g}}}M_\odot$")
axes[0].set(title=r"$N_{\rm halo}$", xlabel="R [cMpc]", ylabel=r"Spearman $\rho$ with $\log M_0$")
axes[1].set(title=r"$M_{\rm halo,tot}$", xlabel="R [cMpc]")
for ax in axes:
    ax.axhline(0, color="k", lw=1)
    ax.set_xticks(R_sphere_cMpc)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Galaxy-threshold dependence

In [ ]:
galaxy_counts, galaxy_mass_sums = multi_threshold_sphere(
    target_pos, sub_pos[galaxy_ids], sub_mstar[galaxy_ids],
    galaxy_ids, target_sub_ids, logMstar_cuts, R_sphere_cMpc, box_size_cMpc,
    query_chunk_size, "Galaxy threshold grid",
)

galaxy_rows = []
for t, threshold in enumerate(logMstar_cuts):
    for r, radius in enumerate(R_sphere_cMpc):
        rho_n, p_n, n_n = safe_spearman(galaxy_counts[:, t, r], targets["logM200c_z0"])
        rho_m, p_m, n_m = safe_spearman(galaxy_mass_sums[:, t, r], targets["logM200c_z0"])
        galaxy_rows.append({
            "log_threshold": threshold, "R_cMpc": radius,
            "rho_count": rho_n, "rho_mass": rho_m,
            "p_count": p_n, "p_mass": p_m, "N": n_n,
            "zero_fraction": np.mean(galaxy_counts[:, t, r] == 0),
            "resolution_caution": threshold < 8.0,
        })
galaxy_correlations = pd.DataFrame(galaxy_rows)
display(galaxy_correlations.round(4))
print("Caution: logMstar_cut=7.5 is a resolution-stress test and is not a complete TNG300 galaxy sample.")

### Galaxy-threshold scale dependence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.7), sharey=True)
for threshold in logMstar_cuts:
    part = galaxy_correlations[galaxy_correlations["log_threshold"] == threshold]
    style = "--" if threshold < 8.0 else "-"
    axes[0].plot(part["R_cMpc"], part["rho_count"], marker="o", ls=style,
                 label=rf"$10^{{{threshold:g}}}M_\odot$")
    axes[1].plot(part["R_cMpc"], part["rho_mass"], marker="o", ls=style,
                 label=rf"$10^{{{threshold:g}}}M_\odot$")
axes[0].set(title=r"$N_{\rm gal}$", xlabel="R [cMpc]", ylabel=r"Spearman $\rho$ with $\log M_0$")
axes[1].set(title=r"$M_{\star,\rm tot}$", xlabel="R [cMpc]")
for ax in axes:
    ax.axhline(0, color="k", lw=1)
    ax.set_xticks(R_sphere_cMpc)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Optimal radius and maximum correlation versus threshold

In [ ]:
optimal_rows = []
for tracer, table in [("halo", halo_correlations), ("galaxy", galaxy_correlations)]:
    for threshold, part in table.groupby("log_threshold"):
        for metric in ["count", "mass"]:
            values = part[f"rho_{metric}"].to_numpy()
            idx = int(np.nanargmax(np.abs(values)))
            chosen = part.iloc[idx]
            optimal_rows.append({
                "Tracer": tracer, "Metric": metric, "log_threshold": threshold,
                "R_opt_cMpc": chosen["R_cMpc"], "rho_max": chosen[f"rho_{metric}"],
                "N": int(chosen["N"]), "zero_fraction_at_opt": chosen["zero_fraction"],
            })
optimal_summary = pd.DataFrame(optimal_rows)
display(optimal_summary.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for (tracer, metric), part in optimal_summary.groupby(["Tracer", "Metric"]):
    label = f"{tracer} {metric}"
    axes[0].plot(part["log_threshold"], part["R_opt_cMpc"], "o-", label=label)
    axes[1].plot(part["log_threshold"], part["rho_max"], "o-", label=label)
axes[0].set(xlabel=r"$\log_{10}$ tracer-mass threshold", ylabel=r"$R_{\rm optimal}$ [cMpc]")
axes[1].set(xlabel=r"$\log_{10}$ tracer-mass threshold", ylabel=r"maximum signed Spearman $\rho$")
axes[1].axhline(0, color="k", lw=1)
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. Observational-cylinder analysis

The fiducial projected tracer is `SubhaloFlag == 1` with $M_\star>10^8M_\odot$. The line of sight is the simulation $z$ axis. Central galaxies are excluded by Subfind ID.

In [ ]:
cyl_mask = np.isfinite(sub_mstar) & (sub_mstar > 10**cylinder_logMstar_cut)
if require_subhalo_flag:
    cyl_mask &= sub_flag
cyl_ids = np.flatnonzero(cyl_mask)

cyl_counts, cyl_mstar_sums = periodic_cylinders(
    target_pos, sub_pos[cyl_ids], sub_mstar[cyl_ids], cyl_ids, target_sub_ids,
    R_proj_cMpc, L_los_cMpc, box_size_cMpc, query_chunk_size, "Projected cylinders",
)

cylinder_rows = []
for l, depth in enumerate(L_los_cMpc):
    for r, radius in enumerate(R_proj_cMpc):
        rho_n, p_n, n = safe_spearman(cyl_counts[:, l, r], targets["logM200c_z0"])
        rho_m, p_m, _ = safe_spearman(cyl_mstar_sums[:, l, r], targets["logM200c_z0"])
        cylinder_rows.append({
            "L_los_cMpc": depth, "R_proj_cMpc": radius,
            "rho_Ngal": rho_n, "rho_MstarTot": rho_m,
            "p_Ngal": p_n, "p_MstarTot": p_m, "N": n,
            "zero_fraction": np.mean(cyl_counts[:, l, r] == 0),
        })
cylinder_correlations = pd.DataFrame(cylinder_rows)
display(cylinder_correlations.round(4))

### Cylinder heatmap and projected-radius curves

In [ ]:
rho_heatmap = cylinder_correlations.pivot(index="L_los_cMpc", columns="R_proj_cMpc", values="rho_Ngal")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
im = axes[0].imshow(rho_heatmap.to_numpy(), origin="lower", aspect="auto", cmap="viridis")
axes[0].set_xticks(np.arange(len(R_proj_cMpc)))
axes[0].set_xticklabels([f"{x:g}" for x in R_proj_cMpc])
axes[0].set_yticks(np.arange(len(L_los_cMpc)))
axes[0].set_yticklabels([f"{x:g}" for x in L_los_cMpc])
axes[0].set(xlabel=r"$R_{\rm proj}$ [cMpc]", ylabel=r"$L_{\rm los}$ [cMpc]",
            title=r"Spearman $\rho(N_{\rm gal,cyl},M_0)$")
fig.colorbar(im, ax=axes[0], label=r"$\rho$")

for depth in L_los_cMpc:
    part = cylinder_correlations[cylinder_correlations["L_los_cMpc"] == depth]
    axes[1].plot(part["R_proj_cMpc"], part["rho_Ngal"], "o-", label=rf"$L_{{los}}={depth:g}$")
axes[1].set(xlabel=r"$R_{\rm proj}$ [cMpc]", ylabel=r"Spearman $\rho$", xticks=R_proj_cMpc)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

### Best cylinder, 3D-sphere comparison, and conditional distributions

In [ ]:
best_cyl_idx = int(np.nanargmax(np.abs(cylinder_correlations["rho_Ngal"])))
best_cylinder = cylinder_correlations.iloc[best_cyl_idx]
best_cyl_depth = float(best_cylinder["L_los_cMpc"])
best_cyl_radius = float(best_cylinder["R_proj_cMpc"])
best_cyl_rho = float(best_cylinder["rho_Ngal"])

star8_index = int(np.flatnonzero(np.isclose(logMstar_cuts, cylinder_logMstar_cut))[0])
sphere_star8 = galaxy_correlations[np.isclose(galaxy_correlations["log_threshold"], cylinder_logMstar_cut)]
best_sphere_idx = int(np.nanargmax(np.abs(sphere_star8["rho_count"])))
best_sphere = sphere_star8.iloc[best_sphere_idx]
best_sphere_radius = float(best_sphere["R_cMpc"])
best_sphere_rho = float(best_sphere["rho_count"])

l_idx = int(np.flatnonzero(np.isclose(L_los_cMpc, best_cyl_depth))[0])
r_idx = int(np.flatnonzero(np.isclose(R_proj_cMpc, best_cyl_radius))[0])
targets["Ngal_cyl_best"] = cyl_counts[:, l_idx, r_idx]
targets["Cylinder_tertile"] = rank_tertiles(targets["Ngal_cyl_best"])

cyl_tertile_rows = []
for label, group in targets.groupby("Cylinder_tertile", observed=True):
    row = percentile_summary(group["logM200c_z0"], str(label))
    row["Ngal_min"] = group["Ngal_cyl_best"].min()
    row["Ngal_max"] = group["Ngal_cyl_best"].max()
    row["P_M0_gt_1e14"] = np.mean(group["M200c_z0"] > 1e14)
    row["P_M0_gt_1e15"] = np.mean(group["M200c_z0"] > 1e15)
    cyl_tertile_rows.append(row)
cylinder_tertile_stats = pd.DataFrame(cyl_tertile_rows)

print(f"Best 3D Ngal sphere: R={best_sphere_radius:g} cMpc, rho={best_sphere_rho:+.3f}")
print(f"Best cylinder: Rproj={best_cyl_radius:g} cMpc, Llos={best_cyl_depth:g} cMpc, rho={best_cyl_rho:+.3f}")
print(f"Projection change in |rho|: {abs(best_cyl_rho)-abs(best_sphere_rho):+.3f}")
display(cylinder_tertile_stats.round(4))

common_bins = np.linspace(targets["logM200c_z0"].min(), targets["logM200c_z0"].max(), 27)
fig, ax = plt.subplots()
for label, group in targets.groupby("Cylinder_tertile", observed=True):
    ax.hist(group["logM200c_z0"], bins=common_bins, density=True, histtype="step", lw=2,
            label=f"{label} cylinder density (N={len(group)})")
ax.set(xlabel=r"$\log_{10}(M_{200c,z=0}^{\rm host}/M_\odot)$", ylabel="Probability density")
ax.legend()
plt.show()

## 7. Lagrangian-radius analysis

Using physical $M_0$ in $M_\odot$ and $\bar\rho_{m,0}=\Omega_m\rho_{\rm crit,0}$ in $M_\odot\,\mathrm{Mpc}^{-3}$ gives a comoving Lagrangian radius in Mpc, directly comparable to the cMpc environment radii:

$$R_{\rm Lag}=\left(3M_0/4\pi\bar\rho_{m,0}\right)^{1/3}.$$

In [ ]:
rho_crit0_msun_mpc3 = 2.77536627e11 * h**2
rho_m0_msun_mpc3 = Omega_m * rho_crit0_msun_mpc3
targets["R_Lag_cMpc"] = (
    3.0 * targets["M200c_z0"] / (4.0*np.pi*rho_m0_msun_mpc3)
)**(1.0/3.0)

rlag16, rlag50, rlag84 = np.percentile(targets["R_Lag_cMpc"], [16, 50, 84])
print(f"rho_crit,0={rho_crit0_msun_mpc3:.4e} Msun/Mpc^3")
print(f"rho_m,0={rho_m0_msun_mpc3:.4e} Msun/Mpc^3")
print(f"R_Lag median={rlag50:.3f} cMpc; 16-84%=[{rlag16:.3f}, {rlag84:.3f}] cMpc")

fig, ax = plt.subplots()
ax.hist(targets["R_Lag_cMpc"], bins=30, alpha=0.8)
ax.axvline(rlag50, color="k", lw=2, label="median R_Lag")
ax.axvline(best_sphere_radius, color="tab:red", ls="--", lw=2, label="best galaxy-sphere R")
ax.set(xlabel=r"$R_{\rm Lag}$ [cMpc]", ylabel="Targets")
ax.legend()
plt.show()

### Lagrangian radius and optimal environment scale by descendant-mass bin

In [ ]:
m0_edges = [-np.inf, 12.5, 13.5, 14.0, 14.5, np.inf]
m0_labels = ["<12.5", "12.5-13.5", "13.5-14.0", "14.0-14.5", ">14.5"]
targets["M0_bin"] = pd.cut(targets["logM200c_z0"], bins=m0_edges, labels=m0_labels, right=False)

# Select the globally strongest spherical observable/threshold for the per-M0-bin scale test.
strongest_opt_idx = int(np.nanargmax(np.abs(optimal_summary["rho_max"])))
strongest_sphere = optimal_summary.iloc[strongest_opt_idx]
strongest_tracer = strongest_sphere["Tracer"]
strongest_metric = strongest_sphere["Metric"]
strongest_threshold = float(strongest_sphere["log_threshold"])
if strongest_tracer == "halo":
    selected_cube = halo_counts if strongest_metric == "count" else halo_mass_sums
    threshold_index = int(np.flatnonzero(np.isclose(logMhalo_cuts, strongest_threshold))[0])
else:
    selected_cube = galaxy_counts if strongest_metric == "count" else galaxy_mass_sums
    threshold_index = int(np.flatnonzero(np.isclose(logMstar_cuts, strongest_threshold))[0])
selected_environment_by_radius = selected_cube[:, threshold_index, :]

m0_bin_rows = []
for label, group in targets.groupby("M0_bin", observed=True):
    idx = group.index.to_numpy()
    rhos = np.array([
        safe_spearman(selected_environment_by_radius[idx, r], group["logM200c_z0"])[0]
        for r in range(len(R_sphere_cMpc))
    ])
    if np.any(np.isfinite(rhos)):
        best_idx = int(np.nanargmax(np.abs(rhos)))
        r_opt, rho_opt = R_sphere_cMpc[best_idx], rhos[best_idx]
    else:
        r_opt, rho_opt = np.nan, np.nan
    q16, q50, q84 = np.percentile(group["R_Lag_cMpc"], [16, 50, 84])
    m0_bin_rows.append({
        "M0_bin": str(label), "N": len(group), "R_Lag16": q16, "R_Lag50": q50,
        "R_Lag84": q84, "R_opt_cMpc": r_opt, "rho_at_Ropt": rho_opt,
    })
m0_bin_summary = pd.DataFrame(m0_bin_rows)
display(m0_bin_summary.round(3))

fig, ax = plt.subplots()
x = np.arange(len(m0_bin_summary))
ax.errorbar(x, m0_bin_summary["R_Lag50"],
            yerr=[m0_bin_summary["R_Lag50"]-m0_bin_summary["R_Lag16"],
                  m0_bin_summary["R_Lag84"]-m0_bin_summary["R_Lag50"]],
            fmt="o", capsize=3, label=r"$R_{\rm Lag}$ median and 16-84%")
ax.plot(x, m0_bin_summary["R_opt_cMpc"], "s--", label=r"environment $R_{\rm optimal}$")
ax.set(xticks=x, xticklabels=m0_bin_summary["M0_bin"], xlabel=r"$\log_{10}(M_0/M_\odot)$ bin",
       ylabel="Comoving scale [cMpc]")
ax.legend()
plt.show()

## 8. Residual and target-mass sanity checks

The linear mean $M_8$–$M_0$ trend is removed before recomputing the scale dependence of the globally strongest environment statistic. Targets are also split into four fixed 0.1-dex $M_8$ bins.

In [ ]:
fit_coeff = np.polyfit(targets["logM200c_z8"], targets["logM200c_z0"], 1)
targets["DeltaLogM0"] = targets["logM200c_z0"] - np.polyval(fit_coeff, targets["logM200c_z8"])
residual_rho = np.array([
    safe_spearman(selected_environment_by_radius[:, r], targets["DeltaLogM0"])[0]
    for r in range(len(R_sphere_cMpc))
])
resid_best_idx = int(np.nanargmax(np.abs(residual_rho)))

m8_edges = np.array([10.8, 10.9, 11.0, 11.1, 11.2])
targets["M8_narrow_bin"] = pd.cut(targets["logM200c_z8"], m8_edges, right=False)
m8_rows = []
for label, group in targets.groupby("M8_narrow_bin", observed=True):
    idx = group.index.to_numpy()
    rhos = np.array([
        safe_spearman(selected_environment_by_radius[idx, r], group["logM200c_z0"])[0]
        for r in range(len(R_sphere_cMpc))
    ])
    best_idx = int(np.nanargmax(np.abs(rhos))) if np.any(np.isfinite(rhos)) else None
    m8_rows.append({
        "M8_bin": str(label), "N": len(group),
        "R_opt_cMpc": R_sphere_cMpc[best_idx] if best_idx is not None else np.nan,
        "rho_max": rhos[best_idx] if best_idx is not None else np.nan,
    })
m8_sanity = pd.DataFrame(m8_rows)

print(f"Strongest sphere statistic: {strongest_tracer} {strongest_metric}, log cut={strongest_threshold:g}")
print(f"Residual best R={R_sphere_cMpc[resid_best_idx]:g} cMpc, rho={residual_rho[resid_best_idx]:+.3f}")
display(m8_sanity.round(3))

fig, ax = plt.subplots()
ax.plot(R_sphere_cMpc, residual_rho, "o-", label=r"correlation with $\Delta\log M_0$")
ax.axhline(0, color="k", lw=1)
ax.set(xlabel="R [cMpc]", ylabel=r"Spearman $\rho$", xticks=R_sphere_cMpc)
ax.legend()
plt.show()

### Central-property comparison and sample-size diagnostics

In [ ]:
rho_m8, _, _ = safe_spearman(targets["logM200c_z8"], targets["logM200c_z0"])
rho_mstar, _, n_mstar = safe_spearman(targets["logMstarCentral_z8"], targets["logM200c_z0"])
rho_sfr, _, n_sfr = safe_spearman(targets["logSFRCentral_z8"], targets["logM200c_z0"])

predictor_comparison = pd.DataFrame([
    {"Predictor": "M8", "Spearman_rho": rho_m8, "N": len(targets)},
    {"Predictor": "central Mstar", "Spearman_rho": rho_mstar, "N": n_mstar},
    {"Predictor": "central SFR", "Spearman_rho": rho_sfr, "N": n_sfr},
    {"Predictor": f"best sphere: {strongest_tracer} {strongest_metric}",
     "Spearman_rho": strongest_sphere["rho_max"], "N": int(strongest_sphere["N"])},
    {"Predictor": "best galaxy cylinder count", "Spearman_rho": best_cyl_rho, "N": int(best_cylinder["N"])},
]).sort_values("Spearman_rho", key=np.abs, ascending=False)
display(predictor_comparison.round(3))

zero_fraction_summary = pd.concat([
    halo_correlations.assign(Tracer="halo"),
    galaxy_correlations.assign(Tracer="galaxy"),
], ignore_index=True)[["Tracer", "log_threshold", "R_cMpc", "N", "zero_fraction"]]
print("Largest zero-neighbor fractions:")
display(zero_fraction_summary.sort_values("zero_fraction", ascending=False).head(12).round(4))

## 9. Save compact numerical summaries

In [ ]:
optimal_summary.to_csv(summary_csv, index=False)
cylinder_correlations.to_csv(Path.cwd()/"tng300_environment_cylinder_correlations.csv", index=False)
m0_bin_summary.to_csv(Path.cwd()/"tng300_environment_lagrangian_summary.csv", index=False)
print(f"Saved: {summary_csv}")
print("Saved cylinder and Lagrangian summary CSV files in the working directory.")

## Automatically generated summary

In [ ]:
strongest_env = optimal_summary.iloc[np.nanargmax(np.abs(optimal_summary["rho_max"]))]
threshold_group = optimal_summary[
    (optimal_summary["Tracer"] == strongest_env["Tracer"])
    & (optimal_summary["Metric"] == strongest_env["Metric"])
]
ropt_span = (threshold_group["R_opt_cMpc"].min(), threshold_group["R_opt_cMpc"].max())
env_beats_central = abs(strongest_env["rho_max"]) > max(abs(rho_mstar), abs(rho_sfr), abs(rho_m8))
projection_loss = abs(best_sphere_rho) - abs(best_cyl_rho)
lag_ratio = float(strongest_env["R_opt_cMpc"] / rlag50)

high_cyl = cylinder_tertile_stats[cylinder_tertile_stats["Label"] == "high"].iloc[0]

display(Markdown(fr"""
### Robustness summary

- The strongest spherical statistic is **{strongest_env['Tracer']} {strongest_env['Metric']}** with tracer threshold
  $10^{{{strongest_env['log_threshold']:g}}}M_\odot$, $R_{{\rm optimal}}={strongest_env['R_opt_cMpc']:g}$ cMpc,
  and Spearman $\rho={strongest_env['rho_max']:+.3f}$.
- For this tracer/metric family, $R_{{\rm optimal}}$ spans **{ropt_span[0]:g}–{ropt_span[1]:g} cMpc** across the tested thresholds;
  the threshold table above shows whether lower-mass tracers systematically improve $|\rho_{{\rm max}}|$.
- The fiducial $M_\star>10^8M_\odot$ 3D galaxy sphere reaches $\rho={best_sphere_rho:+.3f}$ at {best_sphere_radius:g} cMpc.
  The best projected cylinder reaches $\rho={best_cyl_rho:+.3f}$ at
  $(R_{{\rm proj}},L_{{\rm los}})=({best_cyl_radius:g},{best_cyl_depth:g})$ cMpc, changing $|\rho|$ by
  **{-projection_loss:+.3f}** relative to the sphere.
- In the high-cylinder-density tertile, $W_{{68}}={high_cyl['W68']:.3f}$ dex,
  $P(M_0>10^{{14}}M_\odot)={high_cyl['P_M0_gt_1e14']:.2%}$, and
  $P(M_0>10^{{15}}M_\odot)={high_cyl['P_M0_gt_1e15']:.2%}$.
- The descendant Lagrangian radius is $R_{{\rm Lag}}={rlag50:.2f}_{{-{rlag50-rlag16:.2f}}}^{{+{rlag84-rlag50:.2f}}}$ cMpc.
  The strongest spherical $R_{{\rm optimal}}/R_{{\rm Lag,median}}={lag_ratio:.2f}$, so their order-of-magnitude agreement
  can be judged directly rather than assumed.
- The environment {'does' if env_beats_central else 'does not'} outperform narrow-range $M_8$, central stellar mass,
  and central SFR in absolute rank correlation in this realization. The residual test peaks at
  $R={R_sphere_cMpc[resid_best_idx]:g}$ cMpc with $\rho={residual_rho[resid_best_idx]:+.3f}$.
- The $10^{{7.5}}M_\odot$ stellar cut is resolution-limited and should be treated only as a convergence warning.
  High thresholds with large zero-neighbor fractions likewise have reduced rank information.
- For JWST applications, the cylinder experiment is still idealized. The next step is to forward-model redshift errors,
  interlopers, projected masks, luminosity/stellar-mass completeness, and field-to-field selection variation.

These comparisons establish robustness of simulation-derived conditional distributions; they are not causal statements or calibrated Bayesian posteriors.
"""))